# Salinas Spatial Split — SpectralFormer (Official)

This notebook trains and evaluates the **SpectralFormer (Official)** implementation on the Salinas spatial split and performs the final efficiency benchmark.

**Training conditions**
- Spatial split: `seed42`
- Patch size: `15`
- Spectral bands: `204`
- Spectral neighborhood: `3`
- Training batch size: `64`
- Epochs: `50`

**Efficiency benchmark conditions**
- Benchmark batch size: `128`
- Warm-up forward passes: `20`
- Timed forward passes: `100`
- Metrics: parameters, checkpoint size, FLOPs, peak GPU memory, mean/median/P95/std latency, latency per sample, throughput


In [1]:
from pathlib import Path
import sys
import os
import time
from copy import deepcopy
import urllib.request

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from dfm.data.salinas import (
    SALINAS_CLASS_NAMES,
    SalinasPatchDataset,
    load_salinas,
)

from dfm.training.profiling import count_parameters

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

outputs_dir = PROJECT_ROOT / "outputs" / "salinas"
outputs_dir.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Device:", device)


c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_ROOT: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup
Device: cuda


In [2]:
# Reproducibility for this notebook
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)


Seed: 42


In [3]:
# Load the fixed Salinas spatial split
split_path = outputs_dir / "salinas_spatial_split_seed42.npz"

split = np.load(split_path)

train_indices = split["train_indices"]
val_indices = split["val_indices"]
test_indices = split["test_indices"]

print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))


Train: 32337
Validation: 10952
Test: 10840


In [4]:
# Load Salinas
data_dir = PROJECT_ROOT / "data" / "raw" / "salinas"

scene = load_salinas(
    data_dir,
    download=True,
)

print("Cube shape (H, W, C):", scene.cube.shape)
print("Label map shape:", scene.labels.shape)
print("Bands:", scene.bands)
print("Classes:", len(scene.class_names))


Cube shape (H, W, C): (512, 217, 204)
Label map shape: (512, 217)
Bands: 204
Classes: 16


In [5]:
# Base spatial patch datasets
PATCH_SIZE = 15

train_dataset = SalinasPatchDataset(
    scene=scene,
    indices=train_indices,
    patch_size=PATCH_SIZE,
    normalize=True,
)

val_dataset = SalinasPatchDataset(
    scene=scene,
    indices=val_indices,
    patch_size=PATCH_SIZE,
    normalize=True,
)

test_dataset = SalinasPatchDataset(
    scene=scene,
    indices=test_indices,
    patch_size=PATCH_SIZE,
    normalize=True,
)

print(
    "Dataset sizes:",
    len(train_dataset),
    len(val_dataset),
    len(test_dataset),
)


Dataset sizes: 32337 10952 10840


In [6]:
# Official SpectralFormer band grouping
third_party_dir = PROJECT_ROOT / "third_party" / "SpectralFormer"
third_party_dir.mkdir(parents=True, exist_ok=True)

official_vit_path = third_party_dir / "vit_pytorch.py"

if not official_vit_path.exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/danfenghong/IEEE_TGRS_SpectralFormer/main/vit_pytorch.py",
        official_vit_path,
    )

print("SpectralFormer implementation:", official_vit_path)


SpectralFormer implementation: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\third_party\SpectralFormer\vit_pytorch.py


In [7]:
# SpectralFormer requires einops
try:
    from einops import rearrange
    print("einops available.")
except ImportError:
    print("Install einops in the active environment, then rerun this cell.")
    raise


einops available.


In [8]:
sys.path.insert(0, str(third_party_dir))

from vit_pytorch import ViT

print("Loaded:", ViT)


Loaded: <class 'vit_pytorch.ViT'>


In [9]:
def official_band_grouping(
    patch_chw,
    band_patch=3,
):
    """
    SpectralFormer-style spectral neighborhood grouping.

    Input:
        [bands, H, W]

    Output:
        [bands, H*W*band_patch]
    """

    C, H, W = patch_chw.shape

    if band_patch % 2 == 0:
        raise ValueError("band_patch must be odd.")

    x = patch_chw.transpose(1, 2, 0)
    x = x.reshape(H * W, C)

    half = band_patch // 2

    grouped = np.zeros(
        (H * W * band_patch, C),
        dtype=np.float32,
    )

    # Center spectral group
    grouped[
        half * H * W:
        (half + 1) * H * W,
        :
    ] = x

    # Left spectral neighbors
    for i in range(half):
        start = i * H * W
        end = (i + 1) * H * W

        grouped[start:end, :C - i - 1] = x[:, i + 1:]
        grouped[start:end, C - i - 1:] = x[:, :i + 1]

    # Right spectral neighbors
    for i in range(half):
        start = (half + i + 1) * H * W
        end = (half + i + 2) * H * W

        grouped[start:end, :C - i - 1] = x[:, i + 1:]
        grouped[start:end, C - i - 1:] = x[:, :i + 1]

    return grouped.transpose(1, 0).astype(np.float32)


sample_x, sample_y = train_dataset[0]

grouped_x = official_band_grouping(
    sample_x,
    band_patch=3,
)

print("Original:", sample_x.shape)
print("Grouped :", grouped_x.shape)


Original: (204, 15, 15)
Grouped : (204, 675)


In [10]:
class SpectralFormerSalinasDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        base_dataset,
        band_patch=3,
    ):
        self.base_dataset = base_dataset
        self.band_patch = band_patch

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        x, y = self.base_dataset[index]

        x = official_band_grouping(
            x,
            band_patch=self.band_patch,
        )

        return (
            torch.from_numpy(x),
            torch.tensor(y, dtype=torch.long),
        )


BAND_PATCHES = 3

spectralformer_train_dataset = SpectralFormerSalinasDataset(
    train_dataset,
    band_patch=BAND_PATCHES,
)

spectralformer_val_dataset = SpectralFormerSalinasDataset(
    val_dataset,
    band_patch=BAND_PATCHES,
)

spectralformer_test_dataset = SpectralFormerSalinasDataset(
    test_dataset,
    band_patch=BAND_PATCHES,
)

print(
    len(spectralformer_train_dataset),
    len(spectralformer_val_dataset),
    len(spectralformer_test_dataset),
)


32337 10952 10840


In [11]:
# Training loaders
TRAIN_BATCH_SIZE = 64

spectralformer_train_loader = DataLoader(
    spectralformer_train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

spectralformer_val_loader = DataLoader(
    spectralformer_val_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

spectralformer_test_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_sf, y_sf = next(iter(spectralformer_train_loader))

print("Input :", x_sf.shape)
print("Labels:", y_sf.shape)


Input : torch.Size([64, 204, 675])
Labels: torch.Size([64])


In [12]:
# Official SpectralFormer model configuration
spectralformer_model = ViT(
    image_size=15,
    near_band=3,
    num_patches=204,
    num_classes=16,
    dim=64,
    depth=5,
    heads=4,
    mlp_dim=8,
    dropout=0.1,
    emb_dropout=0.1,
    mode="CAF",
).to(device)

print(spectralformer_model)

sf_param_count = count_parameters(spectralformer_model)

print("Trainable parameters:", sf_param_count)
print("Parameters (M):", sf_param_count / 1e6)


ViT(
  (patch_to_embedding): Linear(in_features=675, out_features=64, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-4): 5 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=64, out_features=64, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=64, out_features=8, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=0.1, inplace=False)


In [13]:
# Forward-pass sanity check
x_sf = x_sf.to(device=device, dtype=torch.float32)

with torch.no_grad():
    logits_sf = spectralformer_model(x_sf)

print("Input :", x_sf.shape)
print("Output:", logits_sf.shape)


Input : torch.Size([64, 204, 675])
Output: torch.Size([64, 16])


In [14]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    spectralformer_model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
)


In [15]:
def train_spectralformer_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
):
    model.train()

    running_loss = 0.0
    total = 0

    for x, y in tqdm(loader, leave=False):
        x = x.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.long)

        optimizer.zero_grad(set_to_none=True)

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / total


In [16]:
def evaluate_spectralformer(
    model,
    loader,
    device,
):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for x, y in tqdm(loader, leave=False):
            x = x.to(device, dtype=torch.float32)

            logits = model(x)

            preds = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
    )

    return accuracy, macro_f1, y_true, y_pred


In [17]:
# Train with the existing spatial-split protocol
EPOCHS = 50

best_val_f1 = -1.0
best_epoch = 0
best_state = None

spectralformer_history = []

for epoch in range(1, EPOCHS + 1):

    train_loss = train_spectralformer_epoch(
        spectralformer_model,
        spectralformer_train_loader,
        criterion,
        optimizer,
        device,
    )

    val_acc, val_f1, _, _ = evaluate_spectralformer(
        spectralformer_model,
        spectralformer_val_loader,
        device,
    )

    scheduler.step()

    spectralformer_history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_accuracy": val_acc,
        "val_macro_f1": val_f1,
    })

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = deepcopy(
            spectralformer_model.state_dict()
        )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Val Macro-F1: {val_f1:.4f}"
    )

print()
print("Best epoch:", best_epoch)
print("Best validation Macro-F1:", best_val_f1)


Epoch 01/50 | Loss: 0.3579 | Val Acc: 0.8361 | Val Macro-F1: 0.7835


Epoch 02/50 | Loss: 0.0596 | Val Acc: 0.9200 | Val Macro-F1: 0.8789


Epoch 03/50 | Loss: 0.0313 | Val Acc: 0.9116 | Val Macro-F1: 0.8631


Epoch 04/50 | Loss: 0.0228 | Val Acc: 0.9179 | Val Macro-F1: 0.8768


Epoch 05/50 | Loss: 0.0172 | Val Acc: 0.8650 | Val Macro-F1: 0.7993


...

Epoch 46/50 | Loss: 0.0000 | Val Acc: 0.9254 | Val Macro-F1: 0.8692


Epoch 47/50 | Loss: 0.0000 | Val Acc: 0.9249 | Val Macro-F1: 0.8681


Epoch 48/50 | Loss: 0.0000 | Val Acc: 0.9249 | Val Macro-F1: 0.8687


Epoch 49/50 | Loss: 0.0000 | Val Acc: 0.9252 | Val Macro-F1: 0.8693


Epoch 50/50 | Loss: 0.0000 | Val Acc: 0.9250 | Val Macro-F1: 0.8692

Best epoch: 22
Best validation Macro-F1: 0.8912181029606336


In [18]:
# Restore best validation checkpoint
spectralformer_model.load_state_dict(best_state)
spectralformer_model.eval()


ViT(
  (patch_to_embedding): Linear(in_features=675, out_features=64, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-4): 5 x ModuleList(
        (0): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): Attention(
              (to_qkv): Linear(in_features=64, out_features=192, bias=False)
              (to_out): Sequential(
                (0): Linear(in_features=64, out_features=64, bias=True)
                (1): Dropout(p=0.1, inplace=False)
              )
            )
          )
        )
        (1): Residual(
          (fn): PreNorm(
            (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
            (fn): FeedForward(
              (net): Sequential(
                (0): Linear(in_features=64, out_features=8, bias=True)
                (1): GELU(approximate='none')
                (2): Dropout(p=0.1, inplace=False)


In [19]:
# Save official SpectralFormer checkpoint
spectralformer_path = (
    outputs_dir
    / "spectralformer_official_spatial_best.pt"
)

torch.save(
    {
        "model_state_dict": spectralformer_model.state_dict(),
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "class_names": SALINAS_CLASS_NAMES,
        "patch_size": 15,
        "bands": 204,
        "band_patches": 3,
        "mode": "CAF",
        "seed": 42,
    },
    spectralformer_path,
)

print("Saved:", spectralformer_path)
print("Exists:", spectralformer_path.exists())


Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\spectralformer_official_spatial_best.pt
Exists: True


In [20]:
# Save training history
spectralformer_history_df = pd.DataFrame(
    spectralformer_history
)

history_path = (
    outputs_dir
    / "spectralformer_official_spatial_history.csv"
)

spectralformer_history_df.to_csv(
    history_path,
    index=False,
)

display(spectralformer_history_df.tail())


,epoch,train_loss,val_accuracy,val_macro_f1
45,46,0.000029,0.925402,0.869228
46,47,0.000010,0.924945,0.868082
47,48,0.000008,0.924945,0.868678
48,49,0.000007,0.925219,0.869314
49,50,0.000016,0.925037,0.869235


In [21]:
# Final spatial test evaluation
(
    sf_test_acc,
    sf_test_f1,
    y_test_sf,
    y_pred_sf,
) = evaluate_spectralformer(
    spectralformer_model,
    spectralformer_test_loader,
    device,
)

print("=" * 60)
print("OFFICIAL SPECTRALFORMER SPATIAL TEST RESULTS")
print("=" * 60)
print(f"Accuracy : {sf_test_acc:.4f}")
print(f"Macro-F1 : {sf_test_f1:.4f}")


OFFICIAL SPECTRALFORMER SPATIAL TEST RESULTS
Accuracy : 0.9617
Macro-F1 : 0.9481


In [22]:
# Classification report and confusion matrix
print(
    classification_report(
        y_test_sf,
        y_pred_sf,
        target_names=SALINAS_CLASS_NAMES,
        digits=4,
    )
)

cm_spectralformer = confusion_matrix(
    y_test_sf,
    y_pred_sf,
)

cm_spectralformer_df = pd.DataFrame(
    cm_spectralformer,
    index=SALINAS_CLASS_NAMES,
    columns=SALINAS_CLASS_NAMES,
)

display(cm_spectralformer_df)


                           precision    recall  f1-score   support

    Brocoli_green_weeds_1     0.9912    0.8647    0.9236       909
    Brocoli_green_weeds_2     0.7912    0.9357    0.8574       498
                   Fallow     0.9900    1.0000    0.9950       296
        Fallow_rough_plow     0.9768    1.0000    0.9883       295
            Fallow_smooth     0.9942    0.9965    0.9954       863
                  Stubble     1.0000    1.0000    1.0000       632
                   Celery     0.8054    1.0000    0.8922       120
         Grapes_untrained     0.9649    0.9676    0.9662      2899
     Soil_vinyard_develop     0.9991    0.9890    0.9940      1177
Corn_senesced_green_weeds     0.9495    0.9669    0.9581       544
      Lettuce_romaine_4wk     0.6800    1.0000    0.8095        51
      Lettuce_romaine_5wk     1.0000    1.0000    1.0000       448
      Lettuce_romaine_6wk     0.9209    1.0000    0.9588       233
      Lettuce_romaine_7wk     1.0000    0.9079    0.9517     

,Brocoli_green_weeds_1,Brocoli_green_weeds_2,Fallow,Fallow_rough_plow,Fallow_smooth,Stubble,Celery,Grapes_untrained,Soil_vinyard_develop,Corn_senesced_green_weeds,Lettuce_romaine_4wk,Lettuce_romaine_5wk,Lettuce_romaine_6wk,Lettuce_romaine_7wk,Vinyard_untrained,Vinyard_vertical_trellis
Brocoli_green_weeds_1,786,123,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Brocoli_green_weeds_2,4,466,0,0,0,0,28,0,0,0,0,0,0,0,0,0
Fallow,0,0,296,0,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_rough_plow,0,0,0,295,0,0,0,0,0,0,0,0,0,0,0,0
Fallow_smooth,0,0,0,3,860,0,0,0,0,0,0,0,0,0,0,0
Stubble,0,0,0,0,0,632,0,0,0,0,0,0,0,0,0,0
Celery,0,0,0,0,0,0,120,0,0,0,0,0,0,0,0,0
Grapes_untrained,0,0,0,0,0,0,0,2805,0,7,2,0,20,0,65,0
Soil_vinyard_develop,0,0,0,0,0,0,0,0,1164,11,2,0,0,0,0,0
Corn_senesced_green_weeds,0,0,3,0,1,0,1,2,1,526,10,0,0,0,0,0


In [14]:
# Save spatial test outputs
spectralformer_results = pd.DataFrame([
    {
        "model": "SpectralFormer (Official)",
        "test_accuracy": sf_test_acc,
        "test_macro_f1": sf_test_f1,
    }
])

spectralformer_results_path = (
    outputs_dir
    / "spectralformer_official_spatial_results.csv"
)

spectralformer_results.to_csv(
    spectralformer_results_path,
    index=False,
)

np.save(
    outputs_dir / "spectralformer_official_spatial_y_test.npy",
    y_test_sf,
)

np.save(
    outputs_dir / "spectralformer_official_spatial_y_pred.npy",
    y_pred_sf,
)

cm_spectralformer_df.to_csv(
    outputs_dir
    / "spectralformer_official_spatial_confusion_matrix.csv"
)

display(spectralformer_results)


NameError: name 'sf_test_acc' is not defined

## Final efficiency benchmark

The benchmark below intentionally uses a **separate batch size of 128**, matching the benchmark condition used for the other spatial models.

Training batch size remains 64.

Benchmark protocol:
- batch size: 128
- warm-up: 20 forward passes
- timed runs: 100
- CUDA synchronization around each timed forward pass
- mean, median, P95 and standard deviation
- latency per sample and throughput
- peak GPU memory
- trainable parameters
- checkpoint size
- FLOPs per sample


In [24]:
# Final benchmark configuration
BENCH_BATCH_SIZE = 128
WARMUP_RUNS = 20
TIMED_RUNS = 100

# Dedicated benchmark loader
spectralformer_bench_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=BENCH_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_bench, y_bench = next(
    iter(spectralformer_bench_loader)
)

x_bench = x_bench.to(
    device=device,
    dtype=torch.float32,
)

actual_batch_size = x_bench.shape[0]

print("Benchmark input shape:", tuple(x_bench.shape))
print("Benchmark batch size :", actual_batch_size)

if actual_batch_size != BENCH_BATCH_SIZE:
    print(
        "Note: final test batch may be smaller; "
        "the first benchmark batch is used."
    )


Benchmark input shape: (128, 204, 675)
Benchmark batch size : 128


In [25]:
# Parameters
sf_param_count = count_parameters(
    spectralformer_model
)

sf_parameters_m = sf_param_count / 1e6

print(f"Trainable parameters : {sf_param_count:,}")
print(f"Parameters (M)       : {sf_parameters_m:.6f}")


Trainable parameters : 399,381
Parameters (M)       : 0.399381


In [26]:
# Checkpoint size
if not spectralformer_path.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {spectralformer_path}"
    )

sf_size_mb = (
    os.path.getsize(spectralformer_path)
    / (1024 ** 2)
)

print(
    f"Checkpoint size (MB): "
    f"{sf_size_mb:.6f}"
)


Checkpoint size (MB): 1.552929


In [15]:
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

outputs_dir = PROJECT_ROOT / "outputs" / "salinas"

spectralformer_path = (
    outputs_dir
    / "spectralformer_official_spatial_best.pt"
)

print("Checkpoint:", spectralformer_path)
print("Exists:", spectralformer_path.exists())

Checkpoint: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\spectralformer_official_spatial_best.pt
Exists: True


In [16]:
checkpoint = torch.load(
    spectralformer_path,
    map_location=device,
)

spectralformer_model.load_state_dict(
    checkpoint["model_state_dict"]
)

spectralformer_model = spectralformer_model.to(device)
spectralformer_model.eval()

print("Clean SpectralFormer checkpoint loaded.")

C:\Users\Dines\AppData\Local\Temp\ipykernel_8732\2348824950.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Clean SpectralFormer checkpoint loaded.


In [17]:
from copy import deepcopy
from thop import profile

flops_model = deepcopy(spectralformer_model)
flops_model.eval()

dummy_input = torch.randn(
    1,
    204,
    675,
    device=device,
    dtype=torch.float32,
)

sf_flops, _ = profile(
    flops_model,
    inputs=(dummy_input,),
    verbose=False,
)

sf_gflops = sf_flops / 1e9

print(f"FLOPs / sample : {sf_flops:,.0f}")
print(f"GFLOPs / sample: {sf_gflops:.6f}")

del flops_model
del dummy_input

if torch.cuda.is_available():
    torch.cuda.empty_cache()

FLOPs / sample : 43,319,680
GFLOPs / sample: 0.043320


In [18]:
# Peak GPU memory + benchmark warm-up
spectralformer_model.eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = spectralformer_model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()

if torch.cuda.is_available():
    sf_peak_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
    )
else:
    sf_peak_memory_mb = np.nan

print(
    f"Peak GPU memory: "
    f"{sf_peak_memory_mb:.4f} MB"
)

print(
    f"Warm-up runs   : "
    f"{WARMUP_RUNS}"
)


NameError: name 'WARMUP_RUNS' is not defined

In [19]:
# ============================================================
# Efficiency benchmark configuration
# Same conditions as the previous spatial models
# ============================================================

BENCH_BATCH_SIZE = 128
WARMUP_RUNS = 20
TIMED_RUNS = 100

print("Benchmark batch size:", BENCH_BATCH_SIZE)
print("Warm-up runs:", WARMUP_RUNS)
print("Timed runs:", TIMED_RUNS)

Benchmark batch size: 128
Warm-up runs: 20
Timed runs: 100


In [20]:
# ============================================================
# Create the fixed benchmark batch
# ============================================================

spectralformer_bench_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=BENCH_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_bench, y_bench = next(
    iter(spectralformer_bench_loader)
)

x_bench = x_bench.to(
    device=device,
    dtype=torch.float32,
)

print("Benchmark input shape:", tuple(x_bench.shape))
print("Actual batch size:", x_bench.shape[0])

Benchmark input shape: (128, 204, 675)
Actual batch size: 128


In [21]:
# ============================================================
# Peak GPU memory + warm-up
# ============================================================

spectralformer_model.eval()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():
    for _ in range(WARMUP_RUNS):
        _ = spectralformer_model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()

if torch.cuda.is_available():
    sf_peak_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
    )
else:
    sf_peak_memory_mb = float("nan")

print(f"Peak GPU memory: {sf_peak_memory_mb:.4f} MB")

Peak GPU memory: 325.0615 MB


In [34]:
spectralformer_model.load_state_dict(
    torch.load(
        spectralformer_path,
        map_location=device,
    )["model_state_dict"]
)

spectralformer_model = spectralformer_model.to(device)
spectralformer_model.eval()

print("Model restored.")

C:\Users\Dines\AppData\Local\Temp\ipykernel_30388\2673218495.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


RuntimeError: Error(s) in loading state_dict for ViT:
	Missing key(s) in state_dict: "total_ops", "total_params", "transformer.total_ops", "transformer.total_params", "transformer.layers.total_ops", "transformer.layers.total_params", "transformer.layers.0.total_ops", "transformer.layers.0.total_params", "transformer.layers.0.0.total_ops", "transformer.layers.0.0.total_params", "transformer.layers.0.0.fn.total_ops", "transformer.layers.0.0.fn.total_params", "transformer.layers.0.0.fn.fn.total_ops", "transformer.layers.0.0.fn.fn.total_params", "transformer.layers.0.1.total_ops", "transformer.layers.0.1.total_params", "transformer.layers.0.1.fn.total_ops", "transformer.layers.0.1.fn.total_params", "transformer.layers.0.1.fn.fn.total_ops", "transformer.layers.0.1.fn.fn.total_params", "transformer.layers.0.1.fn.fn.net.1.total_ops", "transformer.layers.0.1.fn.fn.net.1.total_params", "transformer.layers.1.total_ops", "transformer.layers.1.total_params", "transformer.layers.1.0.total_ops", "tr

In [23]:
# ============================================================
# SpectralFormer — FINAL LATENCY / THROUGHPUT BENCHMARK
# Same conditions as previous spatial models
# Batch size = 128
# Warm-up = 20
# Timed runs = 100
# ============================================================

import time
import numpy as np
import torch


# ------------------------------------------------------------
# Benchmark configuration
# ------------------------------------------------------------

BENCH_BATCH_SIZE = 128
WARMUP_RUNS = 20
TIMED_RUNS = 100


# ------------------------------------------------------------
# Create a fixed benchmark batch
# ------------------------------------------------------------

spectralformer_bench_loader = DataLoader(
    spectralformer_test_dataset,
    batch_size=BENCH_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

x_bench, y_bench = next(
    iter(spectralformer_bench_loader)
)

x_bench = x_bench.to(
    device=device,
    dtype=torch.float32,
)

actual_batch_size = x_bench.shape[0]

print("Benchmark input shape:", tuple(x_bench.shape))
print("Actual batch size    :", actual_batch_size)


# ------------------------------------------------------------
# Model in evaluation mode
# ------------------------------------------------------------

spectralformer_model.eval()


# ------------------------------------------------------------
# Warm-up
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)

with torch.no_grad():

    for _ in range(WARMUP_RUNS):
        _ = spectralformer_model(x_bench)

if torch.cuda.is_available():
    torch.cuda.synchronize()


print(f"Warm-up runs         : {WARMUP_RUNS}")


# ------------------------------------------------------------
# Peak GPU memory
# ------------------------------------------------------------

if torch.cuda.is_available():

    sf_peak_memory_mb = (
        torch.cuda.max_memory_allocated(device)
        / (1024 ** 2)
    )

else:

    sf_peak_memory_mb = np.nan


print(
    f"Peak GPU memory      : "
    f"{sf_peak_memory_mb:.4f} MB"
)


# ------------------------------------------------------------
# Timed inference
# ------------------------------------------------------------

sf_latencies = []

with torch.no_grad():

    for _ in range(TIMED_RUNS):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()

        _ = spectralformer_model(x_bench)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

        sf_latencies.append(
            (end - start) * 1000.0
        )


# ------------------------------------------------------------
# Convert timings to NumPy
# ------------------------------------------------------------

sf_latencies = np.asarray(
    sf_latencies,
    dtype=np.float64,
)


# ------------------------------------------------------------
# Latency statistics
# ------------------------------------------------------------

sf_mean_batch_latency = (
    sf_latencies.mean()
)

sf_median_batch_latency = (
    np.median(sf_latencies)
)

sf_p95_batch_latency = (
    np.percentile(
        sf_latencies,
        95,
    )
)

sf_std_batch_latency = (
    sf_latencies.std(
        ddof=1
    )
)


# ------------------------------------------------------------
# Per-sample latency
# ------------------------------------------------------------

sf_latency_per_sample = (
    sf_mean_batch_latency
    / actual_batch_size
)


# ------------------------------------------------------------
# Throughput
# ------------------------------------------------------------

sf_throughput = (
    actual_batch_size
    / (sf_mean_batch_latency / 1000.0)
)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print()
print("=" * 60)
print("SPECTRALFORMER LATENCY RESULTS")
print("=" * 60)

print(
    f"Mean batch latency   : "
    f"{sf_mean_batch_latency:.4f} ms"
)

print(
    f"Median batch latency : "
    f"{sf_median_batch_latency:.4f} ms"
)

print(
    f"P95 batch latency    : "
    f"{sf_p95_batch_latency:.4f} ms"
)

print(
    f"Latency std          : "
    f"{sf_std_batch_latency:.4f} ms"
)

print(
    f"Latency/sample       : "
    f"{sf_latency_per_sample:.6f} ms"
)

print(
    f"Throughput           : "
    f"{sf_throughput:.2f} samples/s"
)

print(
    f"Peak GPU memory      : "
    f"{sf_peak_memory_mb:.4f} MB"
)

print(
    f"Timed runs           : "
    f"{TIMED_RUNS}"
)

Benchmark input shape: (128, 204, 675)
Actual batch size    : 128
Warm-up runs         : 20
Peak GPU memory      : 325.0615 MB

SPECTRALFORMER LATENCY RESULTS
Mean batch latency   : 59.8848 ms
Median batch latency : 59.8221 ms
P95 batch latency    : 61.1018 ms
Latency std          : 0.9260 ms
Latency/sample       : 0.467850 ms
Throughput           : 2137.44 samples/s
Peak GPU memory      : 325.0615 MB
Timed runs           : 100


In [25]:
# ============================================================
# SpectralFormer — FINAL EFFICIENCY RECORD
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. Final spatial test evaluation
# ------------------------------------------------------------

spectralformer_model.eval()

sf_y_true = []
sf_y_pred = []

with torch.no_grad():

    for x, y in spectralformer_test_loader:

        x = x.to(
            device=device,
            dtype=torch.float32,
        )

        logits = spectralformer_model(x)

        predictions = (
            logits
            .argmax(dim=1)
            .cpu()
            .numpy()
        )

        sf_y_pred.extend(predictions)
        sf_y_true.extend(y.numpy())


sf_y_true = np.asarray(sf_y_true)
sf_y_pred = np.asarray(sf_y_pred)


# ------------------------------------------------------------
# 2. Accuracy and Macro-F1
# ------------------------------------------------------------

sf_test_acc = accuracy_score(
    sf_y_true,
    sf_y_pred,
)

sf_test_f1 = f1_score(
    sf_y_true,
    sf_y_pred,
    average="macro",
)


print("=" * 60)
print("FINAL SPATIAL TEST METRICS")
print("=" * 60)

print(
    f"Accuracy : {sf_test_acc:.6f}"
)

print(
    f"Macro-F1 : {sf_test_f1:.6f}"
)


# ------------------------------------------------------------
# 3. Parameter count
# ------------------------------------------------------------

sf_param_count = sum(
    p.numel()
    for p in spectralformer_model.parameters()
    if p.requires_grad
)

sf_parameters_m = (
    sf_param_count / 1e6
)


# ------------------------------------------------------------
# 4. Checkpoint size
# ------------------------------------------------------------

spectralformer_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_best.pt"
)

if spectralformer_path.exists():

    sf_size_mb = (
        spectralformer_path.stat().st_size
        / (1024 ** 2)
    )

else:

    # Fallback to parameter tensor size
    sf_size_mb = (
        sum(
            p.numel() * p.element_size()
            for p in spectralformer_model.parameters()
        )
        / (1024 ** 2)
    )


# ------------------------------------------------------------
# 5. Final efficiency record
# ------------------------------------------------------------

official_sf_metrics_128 = pd.DataFrame([{

    "model":
        "SpectralFormer (Official)",

    "accuracy":
        sf_test_acc,

    "macro_f1":
        sf_test_f1,

    "parameters":
        sf_param_count,

    "parameters_m":
        sf_parameters_m,

    "model_size_mb":
        sf_size_mb,

    "flops":
        sf_flops,

    "gflops":
        sf_gflops,

    "peak_gpu_memory_mb":
        sf_peak_memory_mb,

    "batch_size":
        actual_batch_size,

    "latency_batch_ms":
        sf_mean_batch_latency,

    "latency_batch_median_ms":
        sf_median_batch_latency,

    "latency_batch_p95_ms":
        sf_p95_batch_latency,

    "latency_batch_std_ms":
        sf_std_batch_latency,

    "latency_per_sample_ms":
        sf_latency_per_sample,

    "throughput_samples_sec":
        sf_throughput,

}])


# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

display(
    official_sf_metrics_128
)

FINAL SPATIAL TEST METRICS
Accuracy : 0.961716
Macro-F1 : 0.948078


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.961716,0.948078,399381,0.399381,1.552929,43319680.0,0.04332,325.061523,128,59.884753,59.82215,61.10184,0.926037,0.46785,2137.438891


In [26]:
# ============================================================
# Save final SpectralFormer efficiency benchmark
# ============================================================

sf_efficiency_path = (
    PROJECT_ROOT
    / "outputs"
    / "salinas"
    / "spectralformer_official_spatial_efficiency.csv"
)

official_sf_metrics_128.to_csv(
    sf_efficiency_path,
    index=False,
)

print("Saved:", sf_efficiency_path)
print("Exists:", sf_efficiency_path.exists())


# Verify by reading the actual CSV back
verified_sf = pd.read_csv(
    sf_efficiency_path
)

display(verified_sf)

Saved: c:\Users\Dines\Documents\Codex\2026-05-15\files-mentioned-by-the-user-dataset0\dense-forest-monitoring_backup\outputs\salinas\spectralformer_official_spatial_efficiency.csv
Exists: True


,model,accuracy,macro_f1,parameters,parameters_m,model_size_mb,flops,gflops,peak_gpu_memory_mb,batch_size,latency_batch_ms,latency_batch_median_ms,latency_batch_p95_ms,latency_batch_std_ms,latency_per_sample_ms,throughput_samples_sec
0,SpectralFormer (Official),0.961716,0.948078,399381,0.399381,1.552929,43319680.0,0.04332,325.061523,128,59.884753,59.82215,61.10184,0.926037,0.46785,2137.438891


In [27]:
# Optional: inspect the exact benchmark columns
print("Benchmark columns:")
for column in official_sf_metrics_128.columns:
    print(" -", column)


Benchmark columns:
 - model
 - accuracy
 - macro_f1
 - parameters
 - parameters_m
 - model_size_mb
 - flops
 - gflops
 - peak_gpu_memory_mb
 - batch_size
 - latency_batch_ms
 - latency_batch_median_ms
 - latency_batch_p95_ms
 - latency_batch_std_ms
 - latency_per_sample_ms
 - throughput_samples_sec
